In [3]:
import os, json
from langdetect import detect
import urllib.robotparser
import requests
from bs4 import BeautifulSoup
import json
import wget
from bs4 import BeautifulSoup
import requests
import json
from langdetect import detect
from pdfminer.high_level import extract_text as extract_pdf_text
import mimetypes

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


# Initialiser le corpus et les sources
corpus = []
sources = []

urls = [
    "https://share.google/QbJCh2ygUQQBlGUtb",
    "https://share.google/0sJ6jsMgez1sI3Ye5",
    "https://afristat.org/wp-content/uploads/2022/04/NotesCours_Agri.pdf",
    "https://afristat.org/wp-content/uploads/2022/04/NotesCours_Agri.pdf",
    "https://share.google/JFsAeoTXBfuNtqTEu",
    "https://docs.google.com/forms/d/e/1FAIpQLSduR9VwiyApSfLsXGEu6oklvzNXOkYF-S6m0IV1zu1P3TsDVw/viewform",
     "https://doi.org/10.4060/cd3185en",
    "https://doi.org/10.4060/cd4965en",
    "https://doi.org/10.4060/cd4313en",
    "https://doi.org/10.4060/cc8166en",
    "https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.nitidae.org/files/acf8fe1c/manuel_de_formation_darda.pdf&ved=2ahUKEwipgaTzvdGQAxWBXEEAHaHjC5IQFnoECGsQAQ&sqi=2&usg=AOvVaw1byCUFVDKSINjUz5jsn5KD",
    "https://doi.org/10.4060/cd7304en",
    "https://doi.org/10.4060/cb9479fr",
    "https://openknowledge.fao.org/search?f.isPartOfSeries=La%20situation%20mondiale%20de%20l%27alimentation%20et%20l%27agriculture%20%20(SOFA),equals&spc.sf=dc.date.issued&spc.sd=DESC",
    "https://doi.org/10.4060/cb1447fr"
]


corpus, sources = [], []
pdf_folder = "data/localDocuments"  # Dossier contenant tes PDF
theme = "Agriculture"

def is_allowed(url):
    base_url = url.split("/")[0] + "//" + url.split("/")[2]
    robots_url = base_url + "/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch("*", url)
    except:
        return False 
        
def detectFormat(content_type, url):
    if content_type:
        if "pdf" in content_type:
            return "pdf"
        elif "html" in content_type:
            return "html"
        elif "text/plain" in content_type:
            return "txt"
    ext = mimetypes.guess_type(url)[0]
    if ext:
        if "pdf" in ext:
            return "pdf"
        elif "html" in ext:
            return "html"
        elif "text" in ext:
            return "txt"
    return "unknown"

def extract_text_from_url(url, formatType):
    if formatType == "html":
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        return soup.get_text(separator=' ', strip=True)
    elif formatType == "pdf":
        response = requests.get(url)
        with open("temp.pdf", "wb") as f:
            f.write(response.content)
        return extract_pdf_text("temp.pdf")
    elif formatType == "txt":
        response = requests.get(url)
        return response.text
    else:
        return ""



# Fonction pour vérifier robots.txt
 # Si robots.txt inaccessible, on ne scrape pas

# Scraper les pages autorisées
for url in urls:
    if is_allowed(url):
        logger.info("Autorisé :", url)
        try:
            response = requests.get(url)
            content_type = response.headers.get("Content-Type")
            formatType = detectFormat(content_type, url)
            text = extract_text_from_url(url, formatType)
            if len(text.strip()) > 20:
                langage = detect(text)
                corpus.append({
                    "source": url,
                    "text": text,
                    "metadata": {
                        "format": formatType,
                        "langue": langage,
                        "theme": theme
                    }
                })
                sources.append(url)
            else:
                logger.warning("Texte trop court ou illisible :", url)
        except Exception as e:
            logger.warning("Erreur de scraping :", e)
    else:
        logger.warning("Interdit par robots.txt :", url)





for filename in os.listdir(pdf_folder):
    if filename.lower().endswith(".pdf"):
        path = os.path.join(pdf_folder, filename)
        try:
            text = extract_text(path)
            if len(text.strip()) > 20:
                langage = detect(text)
                corpus.append({
                    "source": path,
                    "text": text,
                    "metadata": {
                        "format": "pdf",
                        "langue": langage,
                        "theme": theme
                    }
                })
            else:
                logger.warning("Texte trop court ou vide :", filename)
        except Exception as e:
            logger.war("Erreur avec", filename, ":", e)

# Sauvegarde
with open("data/corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)



--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\LENOVO T14s\anaconda3\Lib\logging\__init__.py", line 1151, in emit
    msg = self.format(record)
  File "C:\Users\LENOVO T14s\anaconda3\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ~~~~~~~~~~^^^^^^^^
  File "C:\Users\LENOVO T14s\anaconda3\Lib\logging\__init__.py", line 712, in format
    record.message = record.getMessage()
                     ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\LENOVO T14s\anaconda3\Lib\logging\__init__.py", line 400, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\LENOVO T14s\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\LENOVO T14s\AppData\Roaming

FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: 'data/localDocuments'